## Coding GPT-2 LLM architecture from scratch


### Reading raw text 
Dataset used is the Verdict 


In [2]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total characters:", len(raw_text))
print(raw_text[:100])

Total characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


### Use of Regular Expression Library for Separating words

In [3]:
import re

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print(preprocessed[:30])
print("Total tokens:", len(preprocessed))

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
Total tokens: 4690


### Handling Unknown and End-of-File Tokens (`<unk>` and `<eof>`)

In [4]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token: integer for integer, token in enumerate(all_tokens)}

print("Vocab size:", len(vocab))
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

Vocab size: 1132
('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


### Simple Word based tokenizer from scratch uses `re` Library to split words and add `eof` and `unk` tokens

In [5]:
class BasicTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int else "<|unk|>"
            for item in preprocessed
        ]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [6]:
tokenizer = BasicTokenizer(vocab)

text1 = "Hello, my name is Shlok "
text2 = "The palace between the lake in small"
text = " <|endoftext|> ".join((text1, text2))

print("Input text:", text)
ids = tokenizer.encode(text)
print("Encoded IDs:", ids)
print("Decoded text:", tokenizer.decode(ids))

Input text: Hello, my name is Shlok  <|endoftext|> The palace between the lake in small
Encoded IDs: [1131, 5, 697, 1131, 584, 1131, 1130, 93, 1131, 218, 988, 1131, 568, 904]
Decoded text: <|unk|>, my <|unk|> is <|unk|> <|endoftext|> The <|unk|> between the <|unk|> in small


### Using GPT-2 tokenzier which is the Byte Pair Encoder(BPE) - subword tokenization Algorithm based on most common pair of consecutive bytes of data replaced with a byte that does not occer in data

In [8]:
import importlib
import tiktoken
 
print("tiktoken version:", importlib.metadata.version("tiktoken"))
 
tokenizer = tiktoken.get_encoding("gpt2")
 
text = "Hello, my name is Shlok <|endoftext|> The palace between the lake in small"
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print("Encoded:", integers)
print("Decoded:", tokenizer.decode(integers))
 
print("\nUnknown word breakdown:", tokenizer.encode("Akwirw ier"))
print("Decoded back:", tokenizer.decode(tokenizer.encode("Akwirw ier")))
 
encodings = {
    "gpt2": tiktoken.get_encoding("gpt2"),
    "gpt3": tiktoken.get_encoding("p50k_base"),
    "gpt4": tiktoken.get_encoding("cl100k_base"),
}
for model, enc in encodings.items():
    print(f"Vocab size {model.upper()}: {enc.n_vocab}")

tiktoken version: 0.13.0
Encoded: [15496, 11, 616, 1438, 318, 911, 75, 482, 220, 50256, 383, 20562, 1022, 262, 13546, 287, 1402]
Decoded: Hello, my name is Shlok <|endoftext|> The palace between the lake in small

Unknown word breakdown: [33901, 86, 343, 86, 220, 959]
Decoded back: Akwirw ier
Vocab size GPT2: 50257
Vocab size GPT3: 50281
Vocab size GPT4: 100277


### Creating a data loader to create Input-Target Pairs - Using the Sliding Window approach 
LLM prediction task is to predict the next word that follow the input block. The prediction targets are shifted by one. The output of the previous sequence becomes input for the next - an autoregressive training/ Self Supervised training.

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
 
 
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
 
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
 
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
 
    def __len__(self):
        return len(self.input_ids)
 
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
 
 
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return dataloader

In [10]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
 
first_batch = next(data_iter)
print("First batch:", first_batch)
 
second_batch = next(data_iter)
print("Second batch:", second_batch)
 
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

First batch: [tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
Second batch: [tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]
Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


### Creating a Randomly initialized Embedding vector for learning token embeddings
Will be learnt later on.... currently used torch to randomly initialize the embedding matrix 

In [11]:
import torch
 
vocab_size = 6
output_dim = 3
 
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print("Embedding weight matrix:\n", embedding_layer.weight)
 
input_ids = torch.tensor([2, 3, 5, 1])
print("\nEmbeddings for [2,3,5,1]:\n", embedding_layer(input_ids))

Embedding weight matrix:
 Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)

Embeddings for [2,3,5,1]:
 tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


### Added positional embeddings to the the Token embeddings to create the input embeddings given to the LLM
Need to encode positional understanding (for eg. Cat sat on the mat, and On the mat sat the cat.)

In [13]:
vocab_size = 50257
output_dim = 256
 
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
 

max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
 
print("Token IDs shape:", inputs.shape)    
 
token_embeddings = token_embedding_layer(inputs)
print("Token embeddings shape:", token_embeddings.shape) 
 
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print("Positional embeddings shape:", pos_embeddings.shape)  # [4, 256]
 
input_embeddings = token_embeddings + pos_embeddings
print("Final input embeddings shape:", input_embeddings.shape) 

Token IDs shape: torch.Size([8, 4])
Token embeddings shape: torch.Size([8, 4, 256])
Positional embeddings shape: torch.Size([4, 256])
Final input embeddings shape: torch.Size([8, 4, 256])


## Attention Mechanisms
- Self Attention
- Causal (Masked) Self Attention
- Multi-Head Attention

### Implememnting Simple attention matrix with random weights

In [27]:
import torch
import torch.nn as nn
 
 
inputs = torch.tensor([
    [0.12, 0.45, 0.73],  # the
    [0.61, 0.29, 0.55],  # cat
    [0.38, 0.91, 0.14],  # sat
    [0.82, 0.47, 0.60],  # on
    [0.24, 0.68, 0.39],  # a
    [0.57, 0.33, 0.81],  # mat
])
 
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1)
print("Attention weights:\n", attn_weights)
print("Row sums:", attn_weights.sum(dim=-1))
 
all_context_vecs = attn_weights @ inputs
print("Context vectors:\n", all_context_vecs)

Attention weights:
 tensor([[0.1777, 0.1538, 0.1466, 0.1774, 0.1560, 0.1884],
        [0.1460, 0.1702, 0.1413, 0.2095, 0.1392, 0.1938],
        [0.1395, 0.1416, 0.2154, 0.1819, 0.1716, 0.1500],
        [0.1348, 0.1678, 0.1454, 0.2235, 0.1352, 0.1934],
        [0.1575, 0.1481, 0.1821, 0.1795, 0.1660, 0.1668],
        [0.1545, 0.1674, 0.1293, 0.2086, 0.1355, 0.2047]])
Row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
Context vectors:
 tensor([[0.4612, 0.5097, 0.5548],
        [0.4907, 0.5007, 0.5569],
        [0.4609, 0.5515, 0.5074],
        [0.4997, 0.5024, 0.5545],
        [0.4606, 0.5318, 0.5295],
        [0.4901, 0.4934, 0.5668]])


### Self Attention 
 - Makes use 3 weight matrices for Query Key and Value (each trainable) these are multiplied with the input embedding matrix to get the Query Key and Value Matrices
 - Attention weights are then obtained by multiplying Query Matrix with Key Transpose and scaled by root(key_dim) to keep variance near 1 for stable training.
 - This is then passed through softmax activation row-wise, so that for each word we know how much attention to pay to the other words
 - Attention weights are then multiply wit the Values matrix to get the final attention scores

In [16]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2
 
torch.manual_seed(42)
W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
 
    def forward(self, x):
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)
        attn_scores  = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        return attn_weights @ values
 
 
torch.manual_seed(42)
sa = SelfAttention(d_in, d_out)
print("SelfAttention output:\n", sa(inputs))

SelfAttention output:
 tensor([[0.3832, 0.2418],
        [0.3795, 0.2444],
        [0.3816, 0.2425],
        [0.3777, 0.2454],
        [0.3825, 0.2421],
        [0.3796, 0.2444]], grad_fn=<MmBackward0>)


### Causal Attention/ Masked Attention
- weights of future words of the sequence are masked to ensure proper learning and no data leakage
- basically the upper traiangular matrix values after softmax are set to zero and then normalized to prevent data leakage
- this way the current word only sees itself and the words before it in the sequence to predict the next word

In [17]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
 
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)
 
        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        return attn_weights @ values
 
 
batch = torch.stack((inputs, inputs), dim=0)
print("Batch shape:", batch.shape)
 
torch.manual_seed(42)
ca = CausalAttention(d_in, d_out, context_length=inputs.shape[0], dropout=0.0)
context_vecs = ca(batch)
print("CausalAttention output shape:", context_vecs.shape)
print(context_vecs)
 

Batch shape: torch.Size([2, 6, 3])
CausalAttention output shape: torch.Size([2, 6, 2])
tensor([[[0.2896, 0.2528],
         [0.3576, 0.1941],
         [0.3295, 0.2572],
         [0.3795, 0.2451],
         [0.3600, 0.2566],
         [0.3796, 0.2444]],

        [[0.2896, 0.2528],
         [0.3576, 0.1941],
         [0.3295, 0.2572],
         [0.3795, 0.2451],
         [0.3600, 0.2566],
         [0.3796, 0.2444]]], grad_fn=<UnsafeViewBackward0>)


### Multi head Attention
- Multiple Key Query and value matrices(Each set together is a head)
- The outputs (attention scores) from each head are then concatenated(not added) together to get the final score matrx

In [18]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
 
        self.d_out    = d_out
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads
 
        self.W_query  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key    = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout  = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
 
    def forward(self, x):
        b, num_tokens, d_in = x.shape
 
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)
 
        # split into heads: (b, num_tokens, d_out) -> (b, num_heads, num_tokens, head_dim)
        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
 
        attn_scores = queries @ keys.transpose(2, 3)
 
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
 
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
 
        # (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, d_out)
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)
 
 
torch.manual_seed(42)
batch_size, context_length, d_in = batch.shape
d_out = 4  # num_heads=2, so head_dim=2
 
mha = MultiHeadAttention(d_in, d_out, context_length=context_length, dropout=0.0, num_heads=2)
out = mha(batch)
print("MultiHeadAttention output shape:", out.shape)
print(out)

MultiHeadAttention output shape: torch.Size([2, 6, 4])
tensor([[[-0.6019,  0.5486,  0.2190,  0.1456],
         [-0.6173,  0.5729,  0.2424,  0.0744],
         [-0.5727,  0.4823,  0.1598,  0.1331],
         [-0.5992,  0.5083,  0.1806,  0.0958],
         [-0.5859,  0.4896,  0.1644,  0.1180],
         [-0.6032,  0.5157,  0.1871,  0.0979]],

        [[-0.6019,  0.5486,  0.2190,  0.1456],
         [-0.6173,  0.5729,  0.2424,  0.0744],
         [-0.5727,  0.4823,  0.1598,  0.1331],
         [-0.5992,  0.5083,  0.1806,  0.0958],
         [-0.5859,  0.4896,  0.1644,  0.1180],
         [-0.6032,  0.5157,  0.1871,  0.0979]]], grad_fn=<ViewBackward0>)


### Initial GPT-2 Configurations

In [19]:
GPT_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

## Core LLM architecture - The Transformer Block 
This is only a dummy code whith all the important functions decalred in it for now 

#### The important parts
- Layer Norm
- GELU activation
- feed Forward Network
- Skip Connections

In [20]:
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
    def forward(self, x):
        return x
 
class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
    def forward(self, x):
        return x
 
class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb  = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb  = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head   = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
 
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(tok_embeds + pos_embeds)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)
 
 
tokenizer = tiktoken.get_encoding("gpt2")
 
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch = torch.stack([
    torch.tensor(tokenizer.encode(txt1)),
    torch.tensor(tokenizer.encode(txt2)),
], dim=0)
 
torch.manual_seed(42)
dummy_model = DummyGPTModel(GPT_CONFIG)
logits = dummy_model(batch)
print("DummyGPT output shape:", logits.shape)  # [2, 4, 50257]

DummyGPT output shape: torch.Size([2, 4, 50257])


In [21]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps   = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
 
    def forward(self, x):
        mean   = x.mean(dim=-1, keepdim=True)
        var    = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift
 
 
torch.manual_seed(42)
sample = torch.randn(2, 5)
ln = LayerNorm(emb_dim=5)
out = ln(sample)
print("Mean after LN:", out.mean(dim=-1, keepdim=True))
print("Var  after LN:", out.var(dim=-1, keepdim=True, unbiased=False))

Mean after LN: tensor([[-1.1921e-08],
        [ 3.2037e-08]], grad_fn=<MeanBackward1>)
Var  after LN: tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [22]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
 
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))
 
 
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )
 
    def forward(self, x):
        return self.layers(x)
 
 
ffn = FeedForward(GPT_CONFIG)
x = torch.rand(2, 5, 768)
print("FeedForward output shape:", ffn(x).shape)

FeedForward output shape: torch.Size([2, 5, 768])


In [23]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff           = FeedForward(cfg)
        self.norm1        = LayerNorm(cfg["emb_dim"])
        self.norm2        = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
 
    def forward(self, x):
        shortcut = x
        x = self.drop_shortcut(self.att(self.norm1(x)))
        x = x + shortcut
 
        shortcut = x
        x = self.drop_shortcut(self.ff(self.norm2(x)))
        x = x + shortcut
        return x
 
 
torch.manual_seed(42)
x = torch.rand(2, 4, 768)
block = TransformerBlock(GPT_CONFIG)
out = block(x)
print("TransformerBlock — in:", x.shape, "out:", out.shape)
 

TransformerBlock — in: torch.Size([2, 4, 768]) out: torch.Size([2, 4, 768])


In [24]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb  = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb  = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head   = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
 
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(tok_embeds + pos_embeds)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)
 
 
torch.manual_seed(42)
model = GPTModel(GPT_CONFIG)
out = model(batch)
print("GPTModel output shape:", out.shape)  # [2, 4, 50257]
 
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
 
total_params_tied = total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"With weight tying (as in original GPT-2): {total_params_tied:,}")
 
total_mb = total_params * 4 / (1024 ** 2)
print(f"Model size (float32): {total_mb:.2f} MB")
 

GPTModel output shape: torch.Size([2, 4, 50257])
Total parameters: 163,009,536
With weight tying (as in original GPT-2): 124,412,160
Model size (float32): 621.83 MB
